## 1. Partindo de uma coleção buscável

Neste momento, a ingestão não é o foco. Vamos partir de chunks técnicos já preparados, como se eles já tivessem entrado na coleção do sistema.

In [15]:
documents = [
    {
        "id": "Chunk 042",
        "title": "408 Request Timeout em webhooks",
        "text": "408 Request Timeout indica que a entrega do webhook pode ter expirado. Reenvie com segurança quando houver timeout temporário.",
    },
    {
        "id": "Chunk 043",
        "title": "Retentativas seguras",
        "text": "Retry de webhook deve usar retentativa com idempotência para evitar duplicar efeitos quando a operação é reenviada.",
    },
    {
        "id": "Chunk 044",
        "title": "Backoff exponencial",
        "text": "Backoff exponencial reduz pressão sobre serviços instáveis durante falhas temporárias e novas tentativas.",
    },
    {
        "id": "Chunk 077",
        "title": "Idempotência em APIs",
        "text": "Idempotência protege APIs quando uma chamada é enviada mais de uma vez por falha de rede.",
    },
    {
        "id": "Chunk 118",
        "title": "Timeouts em chamadas HTTP",
        "text": "Timeout em chamada HTTP precisa de limite explícito, observabilidade e decisão clara entre cancelar, esperar ou tentar novamente.",
    },
    {
        "id": "Chunk 205",
        "title": "Logs de callbacks assíncronos",
        "text": "Logs de webhook e callbacks ajudam a rastrear entrega, resposta do servidor e erro operacional.",
    },
]

print(f"total de chunks: {len(documents)}")
for document in documents:
    print(f"{document['id']} | {document['title']}")

total de chunks: 6
Chunk 042 | 408 Request Timeout em webhooks
Chunk 043 | Retentativas seguras
Chunk 044 | Backoff exponencial
Chunk 077 | Idempotência em APIs
Chunk 118 | Timeouts em chamadas HTTP
Chunk 205 | Logs de callbacks assíncronos


## 2. Transformando texto em termos

A busca lexical começa quando cada texto vira uma sequência de termos buscáveis. Aqui usamos uma normalização pequena: minúsculas, remoção de acentos e remoção de tokens muito curtos. Termos com um ou dois caracteres ficam fora para reduzir ruído.

In [16]:
import re
import unicodedata

def normalize(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    return "".join(char for char in text if not unicodedata.combining(char))

def tokenize(text):
    terms = re.findall(r"[a-z0-9]+", normalize(text))
    return [term for term in terms if len(term) >= 3]

for document in documents:
    document["terms"] = tokenize(f"{document['title']} {document['text']}")

for document in documents[:2]:
    print(f"{document['id']} -> {document['terms']}")

Chunk 042 -> ['408', 'request', 'timeout', 'webhooks', '408', 'request', 'timeout', 'indica', 'que', 'entrega', 'webhook', 'pode', 'ter', 'expirado', 'reenvie', 'com', 'seguranca', 'quando', 'houver', 'timeout', 'temporario']
Chunk 043 -> ['retentativas', 'seguras', 'retry', 'webhook', 'deve', 'usar', 'retentativa', 'com', 'idempotencia', 'para', 'evitar', 'duplicar', 'efeitos', 'quando', 'operacao', 'reenviada']


## 3. Criando o índice invertido

O índice invertido muda a pergunta operacional. Em vez de começar por cada documento, ele permite sair de um termo e chegar aos chunks onde esse termo aparece.

In [17]:
from collections import Counter, defaultdict

class InvertedIndex:
    def __init__(self):
        self.entries = defaultdict(list)

    def add_document(self, document):
        term_counts = Counter(document["terms"])
        for term, count in sorted(term_counts.items()):
            self.entries[term].append((document["id"], count))

    def find(self, term):
        return self.entries.get(term, [])

def build_inverted_index(documents):
    index = InvertedIndex()
    for document in documents:
        index.add_document(document)
    return index

inverted_index = build_inverted_index(documents)

selected_terms = ["408", "timeout", "webhook", "retry", "erro"]
for term in selected_terms:
    occurrences = ", ".join(
        f"{document_id} ({count}x)"
        for document_id, count in inverted_index.find(term)
    )
    print(f"{term} -> {occurrences}")

408 -> Chunk 042 (2x)
timeout -> Chunk 042 (3x), Chunk 118 (1x)
webhook -> Chunk 042 (1x), Chunk 043 (1x), Chunk 205 (1x)
retry -> Chunk 043 (1x)
erro -> Chunk 205 (1x)


## 4. Vendo as frequências como matriz termo-documento

O índice invertido é a estrutura usada para encontrar candidatos a partir dos termos. A matriz abaixo é uma forma didática de enxergar as mesmas frequências por chunk e por termo antes de calcular os pesos.

In [18]:
def term_document_matrix(documents, terms):
    matrix = []

    for document in documents:
        term_counts = Counter(document["terms"])
        matrix.append([term_counts.get(term, 0) for term in terms])

    return matrix

matrix = term_document_matrix(documents, selected_terms)

print(f"matriz termo-documento: ({len(matrix)}, {len(selected_terms)})")
print(f"{len(matrix)} linhas: chunks buscáveis")
print(f"{len(selected_terms)} colunas: termos selecionados para explicar o mecanismo")
print()
print("documento | " + " | ".join(selected_terms))

for document, row in zip(documents, matrix):
    values = "   |   ".join(str(value) for value in row)
    print(f"{document['id']} | {values}")

matriz termo-documento: (6, 5)
6 linhas: chunks buscáveis
5 colunas: termos selecionados para explicar o mecanismo

documento | 408 | timeout | webhook | retry | erro
Chunk 042 | 2   |   3   |   1   |   0   |   0
Chunk 043 | 0   |   0   |   1   |   1   |   0
Chunk 044 | 0   |   0   |   0   |   0   |   0
Chunk 077 | 0   |   0   |   0   |   0   |   0
Chunk 118 | 0   |   1   |   0   |   0   |   0
Chunk 205 | 0   |   0   |   1   |   0   |   1


## 5. Calculando pesos lexicais

A frequência local não conta a história inteira. O IDF adiciona a raridade global: termos que aparecem em menos chunks tendem a carregar mais poder de diferenciação.

In [19]:
import math

def document_frequency(term, documents):
    return sum(1 for document in documents if term in set(document["terms"]))

def idf(term, documents):
    total_documents = len(documents)
    frequency = document_frequency(term, documents)
    return math.log((total_documents + 1) / (frequency + 1)) + 1

print("termo | chunks com o termo | idf")

for term in selected_terms:
    frequency = document_frequency(term, documents)
    weight = idf(term, documents)
    print(f"{term} | {frequency} | {weight:.2f}")

termo | chunks com o termo | idf
408 | 1 | 2.25
timeout | 2 | 1.85
webhook | 3 | 1.56
retry | 1 | 2.25
erro | 1 | 2.25


## 6. Gerando o ranking lexical

Agora a query também vira termos. O score combina frequência no chunk com raridade na coleção. Para manter o exemplo pequeno, usamos uma versão simplificada de TF-IDF.

In [20]:
def score_document(query, document, documents):
    query_terms = tokenize(query)
    term_counts = Counter(document["terms"])
    total_score = 0
    contributions = []

    for term in query_terms:
        contribution = term_counts.get(term, 0) * idf(term, documents)
        total_score += contribution
        if contribution:
            contributions.append(f"{term}:{contribution:.2f}")
    return total_score, contributions

def search_tfidf(query, documents):
    ranking = []
    for document in documents:
        score, contributions = score_document(query, document, documents)
        ranking.append((score, document, contributions))
    return sorted(ranking, key=lambda item: item[0], reverse=True)


query = "408 timeout webhook retry"
results = search_tfidf(query, documents)

print(f"query: {query}")

for position, (score, document, contributions) in enumerate(results, start=1):
    details = ", ".join(contributions) if contributions else "sem termos da query"
    print(f"#{position} {document['id']} score={score:.2f} ({details})")

query: 408 timeout webhook retry
#1 Chunk 042 score=11.61 (408:4.51, timeout:5.54, webhook:1.56)
#2 Chunk 043 score=3.81 (webhook:1.56, retry:2.25)
#3 Chunk 118 score=1.85 (timeout:1.85)
#4 Chunk 205 score=1.56 (webhook:1.56)
#5 Chunk 044 score=0.00 (sem termos da query)
#6 Chunk 077 score=0.00 (sem termos da query)
